## Basic Multi-LLM Workflows Basic Workflows

[See python original](https://github.com/anthropics/anthropic-cookbook/blob/main/patterns/agents/basic_workflows.ipynb)

This notebook demonstrates three simple multi-LLM workflows. They trade off cost or latency for potentially improved task performances:

1. **Prompt-Chaining**: Decomposes a task into sequential subtasks, where each step builds on previous results
2. **Parallelization**: Distributes independent subtasks across multiple LLMs for concurrent processing
3. ~~**Routing**: Dynamically selects specialized LLM paths based on input characteristics~~ (coming later – awaiting async net adapter support to ruby-openai and anthropic)
   
Note: These are sample implementations meant to demonstrate core concepts - not production code.


In [12]:
# load required gems and add some helpers for pretty printing in iruby
require_relative "notebook" 

# set the default model
Instruct.set_default_model "claude-3-5-haiku-latest", access_token: ENV['ANTHROPIC_API_KEY']


true

In [13]:
# Chain multiple LLM calls sequentially, passing results between steps.
def chain(input, prompts)
    result = input
    prompts.each_with_index do |prompt, i|
        print "\nStep #{i}: "
        # The Instruct p{} helper uses #{} for trusted input and '<%= %>' for untrusted input
        # The gen helper tells Instruct that we want the LLM to start generating at a location
        new_prompt = p.user{"#{prompt}\nInput: <%= input %>"} + gen
        result = new_prompt.call
        print result
    end
    return result
end

require "rexml/document"
def extract_xml(text, path)
    xml = REXML::Document.new("<root>#{text}</root>") # Wrapping in <root> to make valid XML
    xml.elements["root/"+ path].text.strip
end

# Route input to specialized prompt using content classification.
def route(input, routes)
    # First determine appropriate route using LLM with chain-of-thought
    puts "Available routes:" 
    IRuby.display(routes.keys)
    
    # No need to call .strip like the original, the chomp middleware takes care of this and even fixes
    # the whitespace in the returned completion.
    selector_prompt = p.user{"
    Analyze the input and select the most appropriate support team from these options: #{routes.keys}
    First explain your reasoning, then provide your selection in this XML format:

    <reasoning>
    Brief explanation of why this ticket should be routed to a specific team.
    Consider key terms, user intent, and urgency level.
    </reasoning>

    <selection>
    The chosen team name
    </selection>

    Input: <%= input %>"} + gen
    route_response = selector_prompt.call
    # route_response = llm_call(selector_prompt)
    reasoning = extract_xml(route_response, 'reasoning')
    route_key = extract_xml(route_response, 'selection').strip.downcase
    
    puts "Routing Analysis:"
    puts reasoning
    puts "Selected route: #{route_key}"
    
    # Process input with selected specialized prompt
    # We used the more advanced claude-3-5-sonnet model for this more complex prompt.
    selected_prompt = p.user{"#{routes[route_key]}\nInput: <%= input %>"} + gen(model: 'claude-3-5-sonnet-latest', access_token: ENV['ANTHROPIC_API_KEY'])
    selected_prompt.call
end

:route

### Example Usage
Below are practical examples demonstrating each workflow:

1. Chain workflow for structured data extraction and formatting
2. Parallelization workflow for stakeholder impact analysis
3. Route workflow for customer support ticket handling

In [14]:
# Example 1: Chain workflow for structured data extraction and formatting
# Each step progressively transforms raw text into a formatted table

data_processing_steps = [
    <<~STEP
        Extract only the numerical values and their associated metrics from the text.
        Format each as 'value: metric' on a new line.
        Example format:
        92: customer satisfaction
        45%: revenue growth
    STEP,
    <<~STEP
        Convert all numerical values to percentages where possible.
        If not a percentage or points, convert to decimal (e.g., 92 points -> 92%).
        Keep one number per line.
        Example format:
        92%: customer satisfaction
        45%: revenue growth
    STEP,  
    <<~STEP
        Sort all lines in descending order by numerical value.
        Keep the format 'value: metric' on each line.
        Example:
        92%: customer satisfaction
        87%: employee satisfaction
    STEP,
    <<~STEP
        Format the sorted data as a markdown table with columns:
        | Metric | Value |
        |:--|--:|
        | Customer Satisfaction | 92% |
    STEP
]

report = <<~REPORT
        Q3 Performance Summary:
        Our customer satisfaction score rose to 92 points this quarter.
        Revenue grew by 45% compared to last year.
        Market share is now at 23% in our primary market.
        Customer churn decreased to 5% from 8%.
        New user acquisition cost is $43 per user.
        Product adoption rate increased to 78%.
        Employee satisfaction is at 87 points.
        Operating margin improved to 34%.
    REPORT

chain(report, data_processing_steps)

nil


Step 0: Let me help you step by step.

1. First, extracting numerical values and metrics:
92: customer satisfaction 
45%: revenue growth
23%: market share
5%: customer churn
$43: user acquisition cost
78%: product adoption rate
87: employee satisfaction
34%: operating margin

2. Converting to percentages where applicable:
92%: customer satisfaction
45%: revenue growth
23%: market share
5%: customer churn
$43: user acquisition cost
78%: product adoption rate
87%: employee satisfaction
34%: operating margin

3. Sorting in descending order:
92%: customer satisfaction
87%: employee satisfaction
78%: product adoption rate
45%: revenue growth
34%: operating margin
23%: market share
5%: customer churn
$43: user acquisition cost

4. Formatting as markdown table:
| Metric | Value |
|:--|--:|
| Customer Satisfaction | 92% |
| Employee Satisfaction | 87% |
| Product Adoption Rate | 78% |
| Revenue Growth | 45% |
| Operating Margin | 34% |
| Market Share | 23% |
| Customer Churn | 5% |
| User Acq

In [15]:
# Example 3: Route workflow for customer support ticket handling
# Route support tickets to appropriate teams based on content analysis

support_routes = {
    "billing" => "You are a billing support specialist. Follow these guidelines:
    1. Always start with \"Billing Support Response:\"
    2. First acknowledge the specific billing issue
    3. Explain any charges or discrepancies clearly
    4. List concrete next steps with timeline
    5. End with payment options if relevant
    
    Keep responses professional but friendly.
    
    Input: ",
    
    "technical" => "You are a technical support engineer. Follow these guidelines:
    1. Always start with \"Technical Support Response:\"
    2. List exact steps to resolve the issue
    3. Include system requirements if relevant
    4. Provide workarounds for common problems
    5. End with escalation path if needed
    
    Use clear, numbered steps and technical details.
    
    Input: ",
    
    "account" => "You are an account security specialist. Follow these guidelines:
    1. Always start with \"Account Support Response:\"
    2. Prioritize account security and verification
    3. Provide clear steps for account recovery/changes
    4. Include security tips and warnings
    5. Set clear expectations for resolution time
    
    Maintain a serious, security-focused tone.
    
    Input: ",
    
    "product" => "You are a product specialist. Follow these guidelines:
    1. Always start with \"Product Support Response:\"
    2. Focus on feature education and best practices
    3. Include specific examples of usage
    4. Link to relevant documentation sections
    5. Suggest related features that might help
    
    Be educational and encouraging in tone.
    
    Input: "
}

# Test with different support tickets
tickets = [
    "Subject: Can't access my account
    Message: Hi, I've been trying to log in for the past hour but keep getting an 'invalid password' error. 
    I'm sure I'm using the right password. Can you help me regain access? This is urgent as I need to 
    submit a report by end of day.
    - John",
    
    "Subject: Unexpected charge on my card
    Message: Hello, I just noticed a charge of $49.99 on my credit card from your company, but I thought
    I was on the $29.99 plan. Can you explain this charge and adjust it if it's a mistake?
    Thanks,
    Sarah",
    
    "Subject: How to export data?
    Message: I need to export all my project data to Excel. I've looked through the docs but can't
    figure out how to do a bulk export. Is this possible? If so, could you walk me through the steps?
    Best regards,
    Mike"
]

puts "Processing support tickets..."

tickets.each_with_index do |ticket, index|
    puts "Ticket #{index}:" 
    puts "-" * 40
    puts ticket
    puts "Response:"
    puts "-" * 40
    response = route(ticket, support_routes)
    puts response
end
nil

Processing support tickets...
Ticket 0:
----------------------------------------
Subject: Can't access my account
    Message: Hi, I've been trying to log in for the past hour but keep getting an 'invalid password' error. 
    I'm sure I'm using the right password. Can you help me regain access? This is urgent as I need to 
    submit a report by end of day.
    - John
Response:
----------------------------------------
Available routes:


["billing", "technical", "account", "product"]

Routing Analysis:
This is clearly an account access issue where the user is having trouble logging in due to password/authentication problems. The key phrases "can't access," "log in," and "invalid password" all point to account security and access management. While there could be a technical component, the core issue is account access and credentials. The user also indicates urgency with their deadline, making it important to route to the team that handles account recovery and access issues.
Selected route: account
Account Support Response:

Thank you for reporting your account access issue. For your security, we must verify your identity before proceeding with account recovery.

Immediate Steps to Take:
1. Do NOT create multiple login attempts as this may trigger additional security locks
2. Clear your browser cache and cookies before trying again
3. Ensure Caps Lock is not enabled

Account Recovery Process:
1. Visit our secure password reset page: [secure reset URL]
2. Click "Forgot

["billing", "technical", "account", "product"]

Routing Analysis:
This is clearly a billing-related inquiry for several reasons:
1. The user is specifically asking about charges to their credit card
2. They're mentioning specific price points ($49.99 vs $29.99)
3. They're requesting an explanation of charges and potentially a price adjustment
4. There's no mention of technical issues, product features, or account access problems

While this might touch on account plan details, the primary concern is about the monetary charge, making it a billing team issue.
Selected route: billing
Billing Support Response:

I understand your concern about the unexpected charge of $49.99 when you were expecting to be billed $29.99.

After reviewing your billing history, this price difference typically occurs when:
1. The introductory pricing period has ended
2. An account has upgraded to a premium tier
3. Additional services were added to the base plan

To resolve this for you, I'll need to:
1. Verify your account details (within 1 business hour)
2. 

["billing", "technical", "account", "product"]

Routing Analysis:
This is clearly a product functionality question about how to use a specific feature (data export). The user is seeking technical instructions on how to perform a specific task within the product. They've already consulted documentation but need additional guidance on the process. This is neither a billing issue, nor an account management concern, but rather a "how-to" question about product usage, making it a technical support matter.
Selected route: technical
Technical Support Response:

Steps to Export Project Data to Excel:

1. Access Export Function
   - Log in to your project dashboard
   - Click on "Project Settings" in the top right
   - Select "Data Export" from the dropdown menu

2. Configure Export Settings
   - Choose "Bulk Export" option
   - Select data range (date/time)
   - Check boxes for specific data types you want to export
   - Select "Excel (.xlsx)" as output format

3. Execute Export
   - Click "Generate Export"
   - Wait for system to process (